# Track A — training that: `gnina_dense` (val-split, seed 2026)

Notebook nay CHI train 1 minh `gnina_dense` — chay doc lap, song song duoc voi
cac notebook Track A khac (`gnina_dense`, `gnina_default2018`, `pafnucy`)
neu Kaggle cho phep nhieu phien GPU commit cung luc (tai khoan free thuong
gioi han ~2 phien GPU commit dong thoi — neu phien thu 3 bi tu choi
"Max number of running commit GPU sessions is exceeded", doi 1 phien xong
roi chay tiep).

Dung LAI `data/` da co san tu Kaggle Dataset ban da upload (tu file zip
`_output_` cua lan chay geoformerdock) — KHONG chay lai Tier 3/4 (khong tai
lai tu `bits.csb.pitt.edu`), vi 4 model deu phai dung CHINH XAC cung 1 bo
du lieu/split de so sanh cong bang.

**TRUOC KHI CHAY**:
1. Settings (panel phai) -> Accelerator -> GPU T4 x2, Internet -> On (bat tu
   dau, KHONG doi giua chung -- se xoa sach session).
2. Add Data -> gan dung Kaggle Dataset da upload (chua `data/` -- khong can
   `results/` lan nay, model nay train tu dau).

**Neu phien bi Kaggle giet giua chung (het gio, thuong ~9-12h)**: van co the
cuu duoc neu da co checkpoint tot (`results/models/gnina_dense_valsplit_s2026/best_model.pt`)
-- dung `tools/finalize_from_checkpoint.py --model gnina_dense --batch_size <muc da THANH CONG>`
(xem huong dan trong chinh file do) de hoan tat ma khong can train lai.

## 0. Cai dat thu vien truoc tien (khong can restart kernel)

In [ ]:
!pip install -q 'numpy<2' molgrid mlflow
!pip install -q --no-deps pytorch-ignite


In [ ]:
import subprocess, sys
# QUAN TRONG: khong "import numpy/torch" TRUC TIEP o day (Kaggle preload numpy
# rieng vao kernel) — kiem tra qua subprocess. Cung kiem tra torch co phai ban
# CUDA khong: da tung gap that ca "pip install molgrid pytorch-ignite mlflow"
# vo tinh keo theo mot ban torch CPU-only (torch.__version__ += '+cpu' thay vi
# '+cuXXX'), lam moi lenh GPU sau nay loi "CUDA driver version is insufficient"
# ngay tu dau — kiem tra som o day de bao loi trong vai giay thay vi ~9 phut.
r = subprocess.run([sys.executable, '-c', '''
import numpy, torch, molgrid
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
print("torch.cuda.is_available():", torch.cuda.is_available())
print("molgrid: import OK")
assert numpy.__version__.startswith("1."), f"numpy={numpy.__version__} van >=2"
assert torch.cuda.is_available(), (
    f"torch.__version__={torch.__version__} KHONG thay GPU (co the pip install da "
    "vo tinh keo torch ve ban CPU-only, hoac Settings Accelerator chua bat GPU). "
    "Kiem tra Settings panel phai, hoac bao Claude kem dong nay."
)
print("OK")
'''], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr, file=sys.stderr)
    raise RuntimeError('Kiem tra thu vien THAT BAI - dan cho Claude.')


## 1. Lay code moi nhat tu GitHub

In [ ]:
import os
if os.path.isdir('/kaggle/working/VNICT2026_Docking_Paper'):
    !cd /kaggle/working/VNICT2026_Docking_Paper && git pull
else:
    !cd /kaggle/working && git clone https://github.com/ducnm-mimhus/VNICT2026_Docking_Paper.git


In [ ]:
%cd /kaggle/working/VNICT2026_Docking_Paper
!git log --oneline -1


## 2. Tim + copy `data/` tu Kaggle Dataset da attach

Tu dong do trong `/kaggle/input/` (khong can biet ten dataset chinh xac).

In [ ]:
import glob, os, subprocess

print('=== /kaggle/input ===')
print(subprocess.run(['ls', '-la', '/kaggle/input'], capture_output=True, text=True).stdout)

data_candidates = [
    p for p in glob.glob('/kaggle/input/**/data', recursive=True)
    if os.path.isdir(p) and os.path.isdir(os.path.join(p, 'types'))
]
print('data/ candidates:', data_candidates)
assert len(data_candidates) >= 1, (
    'Khong tim thay thu muc data/ (co types/) trong /kaggle/input - '
    'kiem tra lai da Add Data dung dataset chua.'
)
SRC_DATA = data_candidates[0]
print(f'\nDung data/ tu: {SRC_DATA}')


In [ ]:
import shutil, os

DST_DATA = '/kaggle/working/VNICT2026_Docking_Paper/data'
if os.path.isdir(DST_DATA):
    print(f'{DST_DATA} da ton tai - bo qua copy (xoa thu muc do bang tay neu muon copy lai tu dau).')
else:
    print(f'Dang copy {SRC_DATA} -> {DST_DATA} (copy noi bo tren Kaggle, khong qua internet, ~5.5GB, vai phut)...')
    shutil.copytree(SRC_DATA, DST_DATA)
    print('Xong copy data/.')

!du -sh {DST_DATA}


## 3. Track A -- training that: `gnina_dense`

Sieu tham so sao chep NGUYEN VAN tu `train_one()` trong
`scripts/run_overnight_valsplit.sh` (bo phan sourcing conda va wait_for_gpu --
Kaggle da cap san 1 GPU cho phien nay). GPU da bat tu dau (Muc 0 cua notebook,
xem canh bao dau notebook) -- khong can lam gi them o day.

In [ ]:
%%bash
TRAIN_FILE=data/types/ref_uff_train0_split.types
VAL_FILE=data/types/ref_uff_val0.types
TEST_FILE=data/types/ref_uff_test0.types
SEED=2026
MODEL=gnina_dense
OUTDIR=results/models/${MODEL}_valsplit_s${SEED}
LOGFILE=results/logs/${MODEL}_valsplit_s${SEED}.log
mkdir -p results/logs

for bs in 256 128 64; do
    echo "=== ${MODEL}: thu batch_size=${bs} ($(date)) ==="
    rm -rf "${OUTDIR}"
    START=$(date +%s)
    if python -u -m dockbench.training \
        "${TRAIN_FILE}" \
        --testfile "${TEST_FILE}" \
        --valfile "${VAL_FILE}" \
        -d data \
        -m "${MODEL}" \
        --label_pos 0 --affinity_pos 1 \
        --base_lr 0.001 --weight_decay 0.01 \
        --batch_size "${bs}" \
        --random_translation 1.0 --clip_gradients 5.0 \
        -i 100 \
        --iteration_scheme small \
        --lr_dynamic --warmup_epochs 2 \
        --test_every 2 --checkpoint_every 100 \
        --no_roc_auc \
        --scale_affinity_loss 1.0 --delta_affinity_loss 1.0 \
        --scale_ranking 0.05 --ranking_temperature 1.0 --ranking_num_pairs 128 \
        --hard_neg_fraction 0.3 \
        --rank_warmup_epochs 10 --rank_rampup_epochs 15 \
        --scale_pose_coupling 0.00 --lambda_pose 1.2 \
        --pose_warmup_epochs 0 --pose_only_epochs 4 \
        --pose_loss_type focal --pose_focal_gamma 2.0 --pose_focal_alpha 0.75 \
        --pose_class_normalize --pose_balance_batch --pose_balance_target_per_class 32 \
        --pose_prior_logit_scale 0.25 --disable_pose_prior_init \
        --pose_loss_scale 0.5 --pose_total_weight 0.85 --aff_total_weight 0.15 \
        --metric_ema_alpha 0.3 \
        --early_stop_metric composite_cidx_balacc --early_stop_composite_w_cidx 0.5 \
        --early_stop_patience 25 --early_stop_min_delta 0.0001 \
        --scale_dist_constraint 0.02 --scale_anchor_loss 0.01 \
        --normalize_targets --seed "${SEED}" \
        --use_amp \
        -g cuda:0 \
        -o "${OUTDIR}" \
        2>&1 | tee "${LOGFILE}"
    then
        if [ -f "${OUTDIR}/summary.json" ]; then
            ELAPSED=$(( $(date +%s) - START ))
            echo "=== ${MODEL}: THANH CONG voi batch_size=${bs}, mat ${ELAPSED}s (~$(( ELAPSED / 60 )) phut) ==="
            exit 0
        fi
    fi
    echo "=== ${MODEL}: THAT BAI o batch_size=${bs}, thu nho hon ==="
done
echo "=== ${MODEL}: THAT BAI CA 3 MUC batch_size ===" >&2
exit 1


**Neu cell tren bi Kaggle cat giua chung (khong thay dong `THANH CONG`)**:
kiem tra `results/models/gnina_dense_valsplit_s2026/best_model.pt` co ton tai
khong:
```
!ls -la results/models/gnina_dense_valsplit_s2026/
```
Neu co `best_model.pt`, dung `tools/finalize_from_checkpoint.py --model gnina_dense
--batch_size <muc bs da THANH CONG, xem log tren>` de hoan tat khong can train lai
(chi vai phut) -- xem huong dan chi tiet trong `tools/finalize_from_checkpoint.py`
hoac notebook `kaggle_finalize_geoformerdock.ipynb` lam vi du.

## 4. Ket qua -- dan phan nay vao chat cho Claude

In [ ]:
import json
d = json.load(open('results/models/gnina_dense_valsplit_s2026/summary.json'))
print(json.dumps(d, indent=2, ensure_ascii=False))
